 # Step 1: Environment Setup & Installations

In [8]:
# Install required libraries
!pip install --quiet --no-warn-script-location ultralytics opencv-python-headless numpy torch torchvision fastdtw transformers lapx

import sys
import os
import json
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from transformers import CLIPTokenizer, CLIPTextModel
from tqdm.auto import tqdm
import shutil
from google.colab import drive
import subprocess
import datetime

# Add GLA-GCN and TGMF-Pose to the Python path so we can import their models later
if "/content/GLA-GCN" not in sys.path:
    sys.path.append("/content/GLA-GCN")
if "/content/TGMF-Pose" not in sys.path:
    sys.path.append("/content/TGMF-Pose")

# Clone the 3D lifting repositories
if not Path("/content/GLA-GCN").exists():
    !git clone https://github.com/bruceyo/GLA-GCN.git
if not Path("/content/TGMF-Pose").exists():
    !git clone https://github.com/yunduo-vision/TGMF-Pose.git

# Mount Google Drive
drive.mount('/content/drive')

# ---------------------------------------------------------
# UPDATE THIS PATH TO YOUR ACTUAL SHARED DRIVE FOLDER
# ---------------------------------------------------------
BASE_PATH = Path("/content/drive/MyDrive/ITCS 4152 5010")

# Setup directory structure variables
input_dir = BASE_PATH / "Data"
output_video_dir = BASE_PATH / "outputs/annotated_videos"
output_keypoint_dir = BASE_PATH / "outputs/keypoints"
output_standardized_dir = BASE_PATH / "outputs/standardized_2d"
output_glagcn_dir = BASE_PATH / "outputs/poses_3d/glagcn"
output_tgmf_dir = BASE_PATH / "outputs/poses_3d/tgmf_pose"

if not input_dir.exists():
    print(f"ERROR: Cannot find {input_dir}. Check your BASE_PATH.")

# Create directories if they don't exist
# output_video_dir.mkdir(parents=True, exist_ok=True)
# output_keypoint_dir.mkdir(parents=True, exist_ok=True)
# output_standardized_dir.mkdir(parents=True, exist_ok=True)
# output_glagcn_dir.mkdir(parents=True, exist_ok=True)
# output_tgmf_dir.mkdir(parents=True, exist_ok=True)

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device Active: {device}")
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("CRITICAL ERROR: PyTorch cannot see the GPU!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Compute Device Active: cuda
PyTorch Version: 2.10.0+cu128
GPU Device: Tesla T4


 # Step 2: YOLOv8 2D Pose Extraction (Sequential Tracking)

In [14]:
# This cell can take close to 20 minutes to run
# Since new keypoints are saved in output/keypoints
# no need to run again unless you have new videos

RELEVANT_FOLDERS = ["correct", "incorrect"] # Change these to your specific folder names
yolo_model = YOLO("yolov8n-pose.pt")
yolo_model.to('cuda')
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv"}

# Helper for timestamped logs
def log(msg):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}")

log(f"Scanning directory: {input_dir}")

# Filter for only your 2 relevant subdirectories
video_files = [
    p for p in input_dir.rglob("*")
    if p.suffix.lower() in VIDEO_EXTENSIONS and p.parent.name in RELEVANT_FOLDERS
]

if not video_files:
    log(f"ERROR: No videos found in folders {RELEVANT_FOLDERS}. Check your naming!")
else:
    log(f"Found {len(video_files)} relevant videos. Starting pipeline...")

    # Single master progress bar for VSCode stability
    for video_path in tqdm(video_files, desc="Batch Progress"):

        label = video_path.parent.name.lower()
        stem = f"{label}__{video_path.stem}"
        output_keypoint_path = output_keypoint_dir / f"{stem}_keypoints.json"

        # Checkpoint: Skip if done
        if output_keypoint_path.exists():
            continue

        log(f"--- PROCESSING: {stem} ---")

        # A. Copy from Drive (Can take a while for 4K)
        local_video_path = f"/content/temp_{video_path.name}"
        log("  Step 1/4: Copying from Drive to local...")
        shutil.copy(video_path, local_video_path)

        # B. FFMPEG Optimization (The heavy lifting)
        optimized_video_path = f"/content/optimized_{video_path.stem}.mp4"
        log("  Step 2/4: FFMPEG re-encoding (720p @ 15fps)...")

        try:
            subprocess.run([
                "ffmpeg", "-y", "-i", local_video_path,
                "-vf", "scale=-2:720",
                "-r", "15",
                "-c:v", "libx264",
                "-preset", "ultrafast",
                "-an",
                optimized_video_path
            ], check=True, capture_output=True, text=True, timeout=300) # 5-minute timeout
        except subprocess.CalledProcessError as e:
            log(f"  !! FFMPEG FAILED: {stem}")
            log(f"    stdout: {e.stdout}")
            log(f"    stderr: {e.stderr}")
            if os.path.exists(local_video_path): os.remove(local_video_path)
            continue
        except subprocess.TimeoutExpired:
            log(f"  !! FFMPEG TIMED OUT: {stem} after 300 seconds. Skipping this video.")
            if os.path.exists(local_video_path):
                os.remove(local_video_path)
            if os.path.exists(optimized_video_path):
                os.remove(optimized_video_path)
            continue

        # C. Check frames
        cap = cv2.VideoCapture(optimized_video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        if total_frames == 0:
            log(f"  !! EMPTY VIDEO: {stem}. Skipping.")
            os.remove(local_video_path)
            continue

        # D. YOLO GPU Tracking
        log(f"  Step 3/4: YOLO Tracking on GPU ({total_frames} frames)...")

        results_stream = yolo_model.track(
            source=optimized_video_path,
            stream=True,
            persist=True,
            conf=0.3,
            device="cuda:0",
            imgsz=640,
            half=True,
            verbose=False
        )

        all_frames = []
        frame_idx = 0
        missing_frames = 0
        active_track_id = None

        for result in results_stream:
            frame_data = {"frame": frame_idx, "people": []}

            if result.boxes is not None and len(result.boxes) > 0 and result.keypoints is not None:
                xyxy = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                track_ids = result.boxes.id.cpu().numpy() if result.boxes.id is not None else None

                # Hard Attention Logic
                areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
                scores = areas * confs

                primary_index = None
                if scores.size > 0:
                    if active_track_id is not None and track_ids is not None:
                        matches = np.where(track_ids == active_track_id)[0]
                        if matches.size > 0:
                            primary_index = int(matches[0])
                            missing_frames = 0

                    if primary_index is None:
                        if active_track_id is not None:
                            missing_frames += 1
                            if missing_frames >= 10:
                                primary_index = int(np.argmax(scores))
                        else:
                            primary_index = int(np.argmax(scores))

                    if primary_index is not None and track_ids is not None:
                        val = track_ids[primary_index]
                        if not np.isnan(val):
                            active_track_id = int(val)
                else:
                    if active_track_id is not None:
                        missing_frames += 1

                if primary_index is not None:
                    kpts_xy = result.keypoints.xy[primary_index].cpu().numpy()
                    kpts_conf = result.keypoints.conf[primary_index].cpu().numpy() if result.keypoints.conf is not None else np.ones(17)
                    person_data = {"person_id": 0, "keypoints": []}
                    for j_idx, ((x, y), c) in enumerate(zip(kpts_xy, kpts_conf)):
                        person_data["keypoints"].append({"joint_id": j_idx, "x": float(x), "y": float(y), "confidence": float(c)})
                    frame_data["people"].append(person_data)

            all_frames.append(frame_data)
            frame_idx += 1
            # Heartbeat print for long videos
            if frame_idx % 100 == 0:
                print(f"    > Progress: {frame_idx}/{total_frames} frames...")

        # E. Finish up
        with output_keypoint_path.open("w", encoding="utf-8") as f:
            json.dump(all_frames, f, indent=2)

        log("  Step 4/4: JSON saved. Cleaning up local files.")
        os.remove(local_video_path)
        os.remove(optimized_video_path)

[01:09:25] Scanning directory: /content/drive/MyDrive/ITCS 4152 5010/Data
[01:09:25] Found 31 relevant videos. Starting pipeline...


Batch Progress:   0%|          | 0/31 [00:00<?, ?it/s]

[01:09:26] --- PROCESSING: correct__IMG_6995 ---
[01:09:26]   Step 1/4: Copying from Drive to local...
[01:09:26]   Step 2/4: FFMPEG re-encoding (720p @ 15fps)...
[01:11:30]   Step 3/4: YOLO Tracking on GPU (175 frames)...
    > Progress: 100/175 frames...
[01:11:41]   Step 4/4: JSON saved. Cleaning up local files.
[01:11:41] --- PROCESSING: correct__IMG_6989 ---
[01:11:41]   Step 1/4: Copying from Drive to local...
[01:11:42]   Step 2/4: FFMPEG re-encoding (720p @ 15fps)...
[01:14:07]   Step 3/4: YOLO Tracking on GPU (220 frames)...
    > Progress: 100/220 frames...
    > Progress: 200/220 frames...
[01:14:18]   Step 4/4: JSON saved. Cleaning up local files.
[01:14:18] --- PROCESSING: correct__IMG_7003 ---
[01:14:18]   Step 1/4: Copying from Drive to local...
[01:14:18]   Step 2/4: FFMPEG re-encoding (720p @ 15fps)...
[01:15:26]   Step 3/4: YOLO Tracking on GPU (203 frames)...
    > Progress: 100/203 frames...
    > Progress: 200/203 frames...
[01:15:37]   Step 4/4: JSON saved. Cleani

 # Step 3: Data Standardization (JSON -> NPY Tensor)

In [15]:
json_files = list(output_keypoint_dir.rglob("*.json"))
print(f"Found {len(json_files)} JSON files. Standardizing to 3D tensors...")

num_joints = 17
fill = 0.0

for j_path in json_files:
    with j_path.open("r", encoding="utf-8") as f:
        frames = json.load(f)

    sequence = []
    # Loop through the timeline and stack the coordinates
    for frame in frames:
        frame_array = np.full((num_joints, 3), fill, dtype=np.float32)
        people = frame.get("people", [])

        if people and people[0].get("person_id") == 0:
            for joint in people[0].get("keypoints", []):
                j_id = int(joint.get("joint_id", -1))
                if 0 <= j_id < num_joints:
                    frame_array[j_id] = [joint.get("x", fill), joint.get("y", fill), joint.get("confidence", fill)]
        sequence.append(frame_array)

    # Convert the stacked sequence into a single Numpy Array of shape (T, 17, 3)
    npy_array = np.stack(sequence, axis=0).astype(np.float32)

    out_path = output_standardized_dir / f"{j_path.stem}.npy"
    np.save(out_path, npy_array)
    print(f"Standardized {j_path.stem} -> Shape {npy_array.shape}")



Found 31 JSON files. Standardizing to 3D tensors...
Standardized correct__IMG_6995_keypoints -> Shape (175, 17, 3)
Standardized correct__IMG_6989_keypoints -> Shape (220, 17, 3)
Standardized correct__IMG_7003_keypoints -> Shape (203, 17, 3)
Standardized correct__IMG_8555_keypoints -> Shape (67, 17, 3)
Standardized correct__IMG_8564_keypoints -> Shape (35, 17, 3)
Standardized correct__IMG_9774_keypoints -> Shape (131, 17, 3)
Standardized correct__IMG_9776_keypoints -> Shape (124, 17, 3)
Standardized correct__IMG_9795_keypoints -> Shape (94, 17, 3)
Standardized correct__IMG_6992_keypoints -> Shape (151, 17, 3)
Standardized correct__5a10e657-8f8d-4685-a977-d60016b4e7db_keypoints -> Shape (106, 17, 3)
Standardized correct__IMG_7007_keypoints -> Shape (156, 17, 3)
Standardized correct__IMG_8536_keypoints -> Shape (57, 17, 3)
Standardized correct__IMG_6998_keypoints -> Shape (170, 17, 3)
Standardized correct__IMG_6991_keypoints -> Shape (229, 17, 3)
Standardized correct__IMG_8557_keypoints -

 # Step 4: GLA-GCN 3D Lifting Inference

In [17]:
npy_files = list(output_standardized_dir.rglob("*.npy"))
print(f"Running GLA-GCN 3D lifting on {len(npy_files)} standardized files...")

# TODO: Import and initialize the GLA-GCN model from the cloned repository here
# from model.gla_gcn import GLAGCN
# glagcn_model = GLAGCN(num_joints=17, in_channels=3).to(device)
# glagcn_model.load_state_dict(torch.load("path_to_glagcn_weights.pth"))
# glagcn_model.eval()

for npy_path in npy_files:
    sequence_2d = np.load(npy_path)
    T = sequence_2d.shape[0]

    # INLINE LOGIC: Normalize coordinates to the Pelvis (center the skeleton at 0,0)
    normalized_2d = np.copy(sequence_2d)
    for t in range(T):
        # Index 11 = Left Hip, Index 12 = Right Hip
        left_hip, right_hip = sequence_2d[t, 11, :2], sequence_2d[t, 12, :2]
        pelvis = (left_hip + right_hip) / 2.0
        normalized_2d[t, :, :2] = sequence_2d[t, :, :2] - pelvis

    # Convert to PyTorch Tensor: (Batch, Channels, Time, Vertices) -> (1, 3, T, 17)
    tensor_2d = torch.tensor(normalized_2d, dtype=torch.float32).to(device)
    tensor_2d = tensor_2d.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        # TODO: Execute actual forward pass
        # out_3d = glagcn_model(tensor_2d)

        # Simulated tensor output to maintain pipeline flow before model init
        out_3d = torch.zeros((1, 3, T, 17)).to(device)

    # Reformat back to (T, 17, 3) and save
    final_3d_array = out_3d.squeeze(0).permute(1, 2, 0).cpu().numpy()
    save_path = output_glagcn_dir / npy_path.name
    np.save(save_path, final_3d_array)
    print(f"Saved GLA-GCN 3D pose: {save_path.name}")



Running GLA-GCN 3D lifting on 31 standardized files...
Saved GLA-GCN 3D pose: correct__IMG_6995_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_6989_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_7003_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_8555_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_8564_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_9774_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_9776_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_9795_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_6992_keypoints.npy
Saved GLA-GCN 3D pose: correct__5a10e657-8f8d-4685-a977-d60016b4e7db_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_7007_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_8536_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_6998_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_6991_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_8557_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_6997_keypoints.npy
Saved GLA-GCN 3D pose: correct__IMG_7

 # Step 5: TGMF-Pose 3D Lifting Inference

In [18]:
print(f"Running TGMF-Pose 3D lifting on {len(npy_files)} standardized files...")

# TODO: Import and initialize TGMF-Pose model and HuggingFace CLIP here
# from models.tgmf import TGMFOcclusionNet
# tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
# text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
# tgmf_model = TGMFOcclusionNet(num_joints=17).to(device)
# tgmf_model.load_state_dict(torch.load("path_to_tgmf_weights.pth"))
# tgmf_model.eval()

# Define the text prompt for occlusion synthesis
prompt = "A person performing a barbell squat."

for npy_path in npy_files:
    sequence_2d = np.load(npy_path)
    T = sequence_2d.shape[0]

    # INLINE LOGIC: Normalize coordinates to the Pelvis
    normalized_2d = np.copy(sequence_2d)
    for t in range(T):
        left_hip, right_hip = sequence_2d[t, 11, :2], sequence_2d[t, 12, :2]
        pelvis = (left_hip + right_hip) / 2.0
        normalized_2d[t, :, :2] = sequence_2d[t, :, :2] - pelvis

    # Convert to PyTorch Tensor: (1, 3, T, 17)
    tensor_2d = torch.tensor(normalized_2d, dtype=torch.float32).to(device)
    tensor_2d = tensor_2d.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        # TODO: Text Embedding and Forward Pass
        # tokens = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
        # text_embedding = text_model(**tokens).last_hidden_state
        # out_3d = tgmf_model(tensor_2d, text_features=text_embedding)

        # Simulated tensor output
        out_3d = torch.zeros((1, 3, T, 17)).to(device)

    # Reformat back to (T, 17, 3) and save
    final_3d_array = out_3d.squeeze(0).permute(1, 2, 0).cpu().numpy()
    save_path = output_tgmf_dir / npy_path.name
    np.save(save_path, final_3d_array)
    print(f"Saved TGMF-Pose 3D pose: {save_path.name}")

Running TGMF-Pose 3D lifting on 31 standardized files...
Saved TGMF-Pose 3D pose: correct__IMG_6995_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_6989_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_7003_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_8555_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_8564_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_9774_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_9776_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_9795_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_6992_keypoints.npy
Saved TGMF-Pose 3D pose: correct__5a10e657-8f8d-4685-a977-d60016b4e7db_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_7007_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_8536_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_6998_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_6991_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_8557_keypoints.npy
Saved TGMF-Pose 3D pose: correct__IMG_6997_keypoints.npy
Sav